# Exploration de l’API arXiv et parsing PDF

In [ ]:
import requests, re, io
from pypdf import PdfReader
from src.services.chunk import chunk_text

ARXIV_API = "http://export.arxiv.org/api/query"
params = {"search_query": "cat:cs.CL", "max_results": 1}
r = requests.get(ARXIV_API, params=params, timeout=30)
r.raise_for_status()
xml = r.text

entry = xml.split("<entry>")[1]
title = re.search(r"<title>(.*?)</title>", entry, re.S).group(1).strip()
pdf_url = re.search(r'href="(http.*?pdf)"', entry).group(1)
print("Titre:", title)
print("PDF URL:", pdf_url)

pdf = requests.get(pdf_url, timeout=60).content
reader = PdfReader(io.BytesIO(pdf))
full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
print("Extrait du texte:\n", full_text[:600])

chunks = list(chunk_text(full_text, max_chars=1000, overlap=100))
print("Nombre de chunks:", len(chunks))
print("Premier chunk:\n", chunks[0][:600])